### Assignment 6 Beating baselines in a competition

In [65]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import kagglehub

In [9]:
path = "C:\\Users\\dell\\.cache\\kagglehub\\competitions\\catch-me-if-you-can-intruder-detection-through-webpage-session-tracking2"
print(os.listdir(path))

['sample_submission.csv', 'site_dic.pkl', 'test_sessions.csv', 'train.zip', 'train_sessions.csv']


In [11]:
df_train = pd.read_csv(os.path.join(path, 'train_sessions.csv'), index_col='session_id')
df_train.head()

,site1,time1,site2,time2,site3,time3,site4,time4,site5,time5,...,time6,site7,time7,site8,time8,site9,time9,site10,time10,target
session_id,,,,,,,,,,,,,,,,,,,,,
1,718,2014-02-20 10:02:45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,890,2014-02-22 11:19:50,941.0,2014-02-22 11:19:50,3847.0,2014-02-22 11:19:51,941.0,2014-02-22 11:19:51,942.0,2014-02-22 11:19:51,...,2014-02-22 11:19:51,3847.0,2014-02-22 11:19:52,3846.0,2014-02-22 11:19:52,1516.0,2014-02-22 11:20:15,1518.0,2014-02-22 11:20:16,0
3,14769,2013-12-16 16:40:17,39.0,2013-12-16 16:40:18,14768.0,2013-12-16 16:40:19,14769.0,2013-12-16 16:40:19,37.0,2013-12-16 16:40:19,...,2013-12-16 16:40:19,14768.0,2013-12-16 16:40:20,14768.0,2013-12-16 16:40:21,14768.0,2013-12-16 16:40:22,14768.0,2013-12-16 16:40:24,0
4,782,2014-03-28 10:52:12,782.0,2014-03-28 10:52:42,782.0,2014-03-28 10:53:12,782.0,2014-03-28 10:53:42,782.0,2014-03-28 10:54:12,...,2014-03-28 10:54:42,782.0,2014-03-28 10:55:12,782.0,2014-03-28 10:55:42,782.0,2014-03-28 10:56:12,782.0,2014-03-28 10:56:42,0
5,22,2014-02-28 10:53:05,177.0,2014-02-28 10:55:22,175.0,2014-02-28 10:55:22,178.0,2014-02-28 10:55:23,177.0,2014-02-28 10:55:23,...,2014-02-28 10:55:59,175.0,2014-02-28 10:55:59,177.0,2014-02-28 10:55:59,177.0,2014-02-28 10:57:06,178.0,2014-02-28 10:57:11,0


In [13]:
df_train.info()
#pandas NaN olan bir int sütunu otomatik floata çevirir.
#time sütunları str bunları pd.to_datetime() çevirmek gerek.

<class 'pandas.DataFrame'>
RangeIndex: 253561 entries, 1 to 253561
Data columns (total 21 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   site1   253561 non-null  int64  
 1   time1   253561 non-null  str    
 2   site2   250098 non-null  float64
 3   time2   250098 non-null  str    
 4   site3   246919 non-null  float64
 5   time3   246919 non-null  str    
 6   site4   244321 non-null  float64
 7   time4   244321 non-null  str    
 8   site5   241829 non-null  float64
 9   time5   241829 non-null  str    
 10  site6   239495 non-null  float64
 11  time6   239495 non-null  str    
 12  site7   237297 non-null  float64
 13  time7   237297 non-null  str    
 14  site8   235224 non-null  float64
 15  time8   235224 non-null  str    
 16  site9   233084 non-null  float64
 17  time9   233084 non-null  str    
 18  site10  231052 non-null  float64
 19  time10  231052 non-null  str    
 20  target  253561 non-null  int64  
dtypes: float64(9), int64(

In [16]:
df_train['target'].value_counts()
# 0 Alice değil
# 1 Alice 
# Accuracy kullanılamaz — model her şeye "0" dese bile %99 doğruluk elde eder ama hiçbir işe yaramaz. Bu yüzden yarışma ROC-AUC kullanıyor
# (olasılık sıralamasına bakıyor, sınıf dengesizliğine karşı daha dayanıklı).

target
0    251264
1      2297
Name: count, dtype: int64

In [19]:
df_train['site1'].nunique()

15765

In [20]:
site_cols = [f'site{i}' for i in range(1,11)]
all_sites = df_train[site_cols].values.flatten()
print(np.unique(all_sites[~pd.isnull(all_sites)]).shape)

(41601,)


In [22]:
import pickle
with open(os.path.join(path, "site_dic.pkl"), "rb") as f:
    site_dict = pickle.load(f)
print(type(site_dict))
print(len(site_dict))

<class 'dict'>
48371


In [23]:
list(site_dict.items())[:5]

[('www.abmecatronique.com', 25075),
 ('groups.live.com', 13997),
 ('majeureliguefootball.wordpress.com', 42436),
 ('cdt46.media.tourinsoft.eu', 30911),
 ('www.hdwallpapers.eu', 8104)]

In [24]:
id_to_site = {v: k for k, v in site_dict.items()}

In [25]:
alice_sessions = df_train[df_train['target'] == 1]

In [26]:
alice_site_ids = alice_sessions[site_cols].values.flatten()
alice_site_ids = alice_site_ids[~pd.isnull(alice_site_ids)]

In [27]:
from collections import Counter
top_sites = Counter(alice_site_ids).most_common(10)

In [28]:
for site_id, count in top_sites:
    print(id_to_site[int(site_id)], count)

i1.ytimg.com 1382
s.youtube.com 1354
www.youtube.com 1307
www.facebook.com 897
www.google.fr 857
r4---sn-gxo5uxg-jqbe.googlevideo.com 609
apis.google.com 522
r1---sn-gxo5uxg-jqbe.googlevideo.com 522
s.ytimg.com 451
r2---sn-gxo5uxg-jqbe.googlevideo.com 447


In [34]:
df_test = pd.read_csv(os.path.join(path, "test_sessions.csv"), index_col="session_id")
df_test.shape

(82797, 20)

In [31]:
#NaN'ları doldurma
train_test_sites = pd.concat([df_train[site_cols], df_test[site_cols]])
train_test_sites = train_test_sites.fillna(0).astype('int')

In [32]:
sites_flatten = train_test_sites[site_cols].values
sites_as_strings = [' '.join(map(str, row)) for row in sites_flatten]

In [35]:
print(len(sites_as_strings))
print(sites_as_strings[0])
print(sites_as_strings[1])

336358
718 0 0 0 0 0 0 0 0 0
890 941 3847 941 942 3846 3847 3846 1516 1518


In [37]:
cv = CountVectorizer()
sites_sparse = cv.fit_transform(sites_as_strings)

In [39]:
print(sites_sparse.shape)

(336358, 48362)


In [40]:
print('0' in cv.vocabulary_)

False


In [41]:
# Örneğin site ID '7' var mı orijinal veride?
print(any(train_test_sites[site_cols].values.flatten() == 7))
# Sözlükte '7' var mı?
print('7' in cv.vocabulary_)

True
False


In [42]:
cv = CountVectorizer(token_pattern=r'\d+')
sites_sparse = cv.fit_transform(sites_as_strings)
print(sites_sparse.shape)
#\d+ demek "bir veya daha fazla rakam" — artık tek haneli sayılar da (0 dahil) token olarak sayılacak.

(336358, 48372)


In [43]:
X_train_sites = sites_sparse[:df_train.shape[0], :]
X_test_sites = sites_sparse[df_train.shape[0]:, :]

print(X_train_sites.shape)
print(X_test_sites.shape)

(253561, 48372)
(82797, 48372)


In [44]:
time_cols = [f'time{i}' for i in range(1, 11)]

for col in time_cols:
    df_train[col] = pd.to_datetime(df_train[col])
    df_test[col] = pd.to_datetime(df_test[col])

In [45]:
df_train['time1'].dtype

dtype('<M8[us]')

In [46]:
df_train['hour'] = df_train['time1'].dt.hour
df_test['hour'] = df_test['time1'].dt.hour

df_train['hour'].value_counts().sort_index()

hour
7       341
8     25369
9     31741
10    33676
11    30798
12    17420
13    22552
14    27306
15    21640
16    18739
17    12830
18     3898
19     1540
20     1200
21     1705
22     1467
23     1339
Name: count, dtype: int64

In [47]:
alice_hours = df_train[df_train['target'] == 1]['hour'].value_counts().sort_index()
others_hours = df_train[df_train['target'] == 0]['hour'].value_counts(normalize=True).sort_index()
alice_hours_norm = df_train[df_train['target'] == 1]['hour'].value_counts(normalize=True).sort_index()

print("Alice (oran):")
print(alice_hours_norm)
print("\nDiğerleri (oran):")
print(others_hours)

Alice (oran):
hour
9     0.016543
11    0.001306
12    0.148019
13    0.085329
14    0.001741
15    0.017414
16    0.382673
17    0.269047
18    0.077928
Name: proportion, dtype: float64

Diğerleri (oran):
hour
7     0.001357
8     0.100966
9     0.126174
10    0.134026
11    0.122560
12    0.067976
13    0.088974
14    0.108659
15    0.085965
16    0.071081
17    0.048602
18    0.014801
19    0.006129
20    0.004776
21    0.006786
22    0.005838
23    0.005329
Name: proportion, dtype: float64


In [48]:
df_train['weekday'] = df_train['time1'].dt.weekday
df_test['weekday'] = df_test['time1'].dt.weekday

alice_wd = df_train[df_train['target'] == 1]['weekday'].value_counts(normalize=True).sort_index()
others_wd = df_train[df_train['target'] == 0]['weekday'].value_counts(normalize=True).sort_index()

print("Alice:\n", alice_wd)
print("\nDiğerleri:\n", others_wd)

Alice:
 weekday
0    0.381367
1    0.221158
2    0.016543
3    0.212016
4    0.150631
5    0.017849
6    0.000435
Name: proportion, dtype: float64

Diğerleri:
 weekday
0    0.157750
1    0.191635
2    0.222607
3    0.173761
4    0.162355
5    0.062715
6    0.029176
Name: proportion, dtype: float64


In [50]:
scaler = StandardScaler()

train_features_num = df_train[['hour', 'weekday']].values
test_features_num = df_test[['hour', 'weekday']].values

train_features_num_scaled = scaler.fit_transform(train_features_num)
test_features_num_scaled = scaler.transform(test_features_num)

In [52]:
X_train = hstack([X_train_sites, train_features_num_scaled])
X_test = hstack([X_test_sites, test_features_num_scaled])

print(X_train.shape)
print(X_test.shape)

(253561, 48374)
(82797, 48374)


In [54]:
y = df_train['target'].values

X_train_part, X_valid, y_train_part, y_valid = train_test_split(
    X_train, y, test_size=0.3, random_state=17, stratify=y
)

In [56]:
logit = LogisticRegression(random_state=17, solver='liblinear')
logit.fit(X_train_part, y_train_part)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",17
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setti

In [58]:
valid_pred = logit.predict_proba(X_valid)[:, 1]
roc_auc_score(y_valid, valid_pred)

0.9737630548038944

In [59]:
logit_full = LogisticRegression(random_state=17, solver='liblinear')
logit_full.fit(X_train, y)

test_pred = logit_full.predict_proba(X_test)[:, 1]

In [60]:
print(test_pred.shape)
print(test_pred[:5])

(82797,)
[3.40336580e-04 8.98815613e-08 4.42479774e-09 1.17827582e-08
 1.68714682e-05]


In [61]:
sample_sub = pd.read_csv(os.path.join(path, "sample_submission.csv"))
sample_sub.head()

,session_id,target
0,1,0.948255
1,2,0.682483
2,3,0.502855
3,4,0.345556
4,5,0.889428


In [62]:
submission = pd.DataFrame({
    'session_id': df_test.index,
    'target': test_pred
})

submission.head()

,session_id,target
0,1,3.403366e-04
1,2,8.988156e-08
2,3,4.424798e-09
3,4,1.178276e-08
4,5,1.687147e-05


In [63]:
submission.to_csv('submission.csv', index=False)

In [64]:
print(os.path.abspath('submission.csv'))

C:\Users\dell\mlcourse.ai\submission.csv


In [67]:
!kaggle competitions submit -c catch-me-if-you-can-intruder-detection-through-webpage-session-tracking2 -f submission.csv -m "Logistic Regression + site bag-of-words + hour/weekday"

99 submissions remaining today.
Successfully submitted to Catch Me If You Can ("Alice")



  0%|          | 0.00/2.28M [00:00<?, ?B/s]
  1%|          | 16.0k/2.28M [00:00<00:57, 41.2kB/s]
  8%|8         | 192k/2.28M [00:00<00:04, 472kB/s]  
 21%|##1       | 496k/2.28M [00:00<00:01, 1.16MB/s]
 29%|##9       | 688k/2.28M [00:01<00:02, 729kB/s] 
 37%|###7      | 864k/2.28M [00:01<00:01, 893kB/s]
 43%|####3     | 0.98M/2.28M [00:01<00:01, 934kB/s]
 49%|####8     | 1.11M/2.28M [00:01<00:01, 620kB/s]
 59%|#####8    | 1.34M/2.28M [00:01<00:01, 886kB/s]
 65%|######5   | 1.48M/2.28M [00:02<00:01, 808kB/s]
 75%|#######4  | 1.70M/2.28M [00:02<00:00, 1.06MB/s]
 82%|########1 | 1.86M/2.28M [00:02<00:00, 958kB/s] 
 88%|########8 | 2.02M/2.28M [00:02<00:00, 1.07MB/s]
 95%|#########4| 2.16M/2.28M [00:02<00:00, 1.11MB/s]
100%|##########| 2.28M/2.28M [00:03<00:00, 735kB/s] 


In [68]:
!kaggle competitions submissions -c catch-me-if-you-can-intruder-detection-through-webpage-session-tracking2

     ref  fileName        date                        description                                             status                     publicScore  privateScore  
--------  --------------  --------------------------  ------------------------------------------------------  -------------------------  -----------  ------------  
55840593  submission.csv  2026-08-28 09:24:47.223000  Logistic Regression + site bag-of-words + hour/weekday  SubmissionStatus.COMPLETE  0.91934      0.93048       
